In [ ]:
%pip -q install fastapi uvicorn pytest httpx ruff mypy bandit pip-audit

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
(ROOT / "tests").mkdir(exist_ok=True)
(ROOT / ".github" / "workflows").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Rezervacije API", version="1.0.0")

class ZahtjevCijene(BaseModel):
    noci: int = Field(ge=1, le=30)
    cijena_noci: float = Field(gt=0, le=10000)
    popust: float = Field(default=0.0, ge=0.0, le=1.0)

class OdgovorCijene(BaseModel):
    iznos: float
    valuta: str = "EUR"

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/cijena", response_model=OdgovorCijene)
def izracunaj_cijenu(z: ZahtjevCijene):
    iznos = round(z.noci * z.cijena_noci * (1 - z.popust), 2)
    return OdgovorCijene(iznos=iznos)

@app.get("/info")
def info():
    return {"naziv": app.title, "verzija": app.version}


In [ ]:
%%writefile tests/test_main.py
import sys
from pathlib import Path

import pytest
from fastapi.testclient import TestClient

# Omoguci import main.py iz korijenskog direktorijuma projekta u Colabu.
PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
from main import app

client = TestClient(app)

@pytest.fixture
def zahtjev():
    return {"noci": 3, "cijena_noci": 80.0, "popust": 0.0}

def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json() == {"status": "ok"}

def test_info():
    response = client.get("/info")
    assert response.status_code == 200
    assert response.json()["verzija"] == "1.0.0"

def test_cijena_standard(zahtjev):
    response = client.post("/cijena", json=zahtjev)
    assert response.status_code == 200
    assert response.json()["iznos"] == 240.0
    assert response.json()["valuta"] == "EUR"

@pytest.mark.parametrize("noci,cijena,popust,expected", [
    (1, 100, 0, 100), (2, 50, 0.5, 50),
    (3, 80, 1, 0), (7, 99.99, 0.1, 629.94),
])
def test_cijena_parametrizovana(noci, cijena, popust, expected):
    response = client.post("/cijena", json={
        "noci": noci, "cijena_noci": cijena, "popust": popust
    })
    assert response.status_code == 200
    assert response.json()["iznos"] == pytest.approx(expected)

@pytest.mark.parametrize("payload", [
    {"noci": 0, "cijena_noci": 80}, {"noci": 31, "cijena_noci": 80},
    {"noci": 2, "cijena_noci": 0}, {"noci": 2, "cijena_noci": -5},
    {"noci": 2, "cijena_noci": 80, "popust": -0.1},
    {"noci": 2, "cijena_noci": 80, "popust": 1.1},
    {"noci": "tri", "cijena_noci": 80}, {},
])
def test_nevalidan_unos(payload):
    assert client.post("/cijena", json=payload).status_code == 422

def test_unknown_route():
    assert client.get("/nepostojeci").status_code == 404


In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
pytest
httpx
ruff
mypy
bandit
pip-audit


In [ ]:
%%writefile .github/workflows/ci.yml
name: CI
on: [push, pull_request]
jobs:
  quality:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -r requirements.txt
      - run: ruff check main.py tests
      - run: mypy main.py
      - run: pytest -q
      - run: bandit -r main.py
      - run: pip-audit


In [ ]:
# Pokreni testove iz direktorijuma projekta
from pathlib import Path
print('Radni direktorijum:', Path.cwd())
print('main.py postoji:', Path('main.py').exists())
print('testovi postoje:', Path('tests/test_main.py').exists())
assert Path('main.py').exists(), 'Prvo pokreni ćeliju koja kreira main.py'
assert Path('tests/test_main.py').exists(), 'Prvo pokreni ćeliju koja kreira tests/test_main.py'
!python -m pytest -q tests/test_main.py


In [ ]:
# Provjera stila i bezbjednosti
!ruff check main.py tests
!bandit -r main.py
